In [1]:
from pathlib import Path

project_dir = Path.cwd()

# Check project files rather than requiring a particular folder name.
required_files = [
    "README.md",
    "01_retrieval_baseline.ipynb",
]

if not all((project_dir / name).is_file() for name in required_files):
    raise RuntimeError(
        f"Launch this notebook from the project root: {project_dir}"
    )

data_dir = project_dir / "data" / "synthetic"
data_dir.mkdir(parents=True, exist_ok=True)

# Original, fictional policies created for this training lab.
documents = {
    "leave_policy.txt": """\
SYNTHETIC TRAINING DOCUMENT
Company: Example Workplace
Document ID: HR-001
Title: Annual Leave Policy

Full-time employees receive 20 days of annual leave per calendar year.
Annual leave requests must be submitted through the employee portal.
The employee's manager must approve the request before leave begins.
""",
    "remote_work_policy.txt": """\
SYNTHETIC TRAINING DOCUMENT
Company: Example Workplace
Document ID: HR-002
Title: Remote Work Policy

Eligible employees may work remotely up to two days per week.
The employee's manager must approve the remote work arrangement.
This policy does not specify reimbursement for home internet expenses.
""",
    "it_support_policy.txt": """\
SYNTHETIC TRAINING DOCUMENT
Company: Example Workplace
Document ID: IT-001
Title: IT Support Policy

Employees must report lost company laptops to the IT service desk immediately.
The IT service desk operates Monday through Friday, from 09:00 to 18:00.
Employees must not share passwords with colleagues or IT support personnel.
""",
}

# Preserve existing files if this cell is run again.
for filename, content in documents.items():
    file_path = data_dir / filename

    if file_path.exists():
        if file_path.read_text(encoding="utf-8") != content:
            raise RuntimeError(
                f"Existing content differs: {filename}. Review it before continuing."
            )
        print(f"Already exists: {filename}")
    else:
        file_path.write_text(content, encoding="utf-8")
        print(f"Created: {filename}")

print(f"\nData folder: {data_dir}")
print(f"Documents verified: {len(documents)}")

Already exists: leave_policy.txt
Already exists: remote_work_policy.txt
Already exists: it_support_policy.txt

Data folder: C:\Users\Khaleel\workplace-ai-operations-lab\data\synthetic
Documents verified: 3


In [2]:
from pathlib import Path
import re

data_dir = Path.cwd() / "data" / "synthetic"

# Common words excluded from our simple matching method.
STOP_WORDS = {
    "a", "an", "the", "is", "are", "do", "does", "how",
    "what", "which", "of", "to", "for", "per", "in",
    "on", "and", "or", "with",
}

def tokenize(text):
    """Return unique lowercase words, excluding common words."""
    words = re.findall(r"[a-z0-9]+", text.lower())
    return set(words) - STOP_WORDS


# Index each policy sentence separately.
# Skip the document headers by reading after the first blank line.
passages = []

for file_path in sorted(data_dir.glob("*.txt")):
    text = file_path.read_text(encoding="utf-8")
    header, separator, body = text.partition("\n\n")

    if not separator:
        raise ValueError(f"Missing header/body separator: {file_path.name}")

    for line_number, line in enumerate(body.splitlines(), start=1):
        sentence = line.strip()
        if sentence:
            passages.append({
                "source": file_path.name,
                "passage_id": f"{file_path.stem}:{line_number}",
                "text": sentence,
                "tokens": tokenize(sentence),
            })

if not passages:
    raise RuntimeError("No policy passages were found.")


def retrieve(question, top_k=2):
    """Rank passages by the number of unique words shared with the question."""
    question_tokens = tokenize(question)
    results = []

    for passage in passages:
        matched_words = question_tokens & passage["tokens"]

        if matched_words:
            results.append({
                "source": passage["source"],
                "passage_id": passage["passage_id"],
                "text": passage["text"],
                "score": len(matched_words),
                "matched_words": sorted(matched_words),
            })

    # Resolve score ties consistently using the passage ID.
    results.sort(key=lambda item: (-item["score"], item["passage_id"]))
    return results[:top_k]


questions = [
    "How many annual leave days do full-time employees receive?",
    "What is the monthly home internet reimbursement amount?",
    "How many annual leave days do contractors receive?",
]

print(f"Indexed passages: {len(passages)}")

for question in questions:
    print("\n" + "=" * 70)
    print("QUESTION:", question)

    results = retrieve(question)

    if not results:
        print("No matching passages.")
        continue

    for rank, result in enumerate(results, start=1):
        print(f"\nRank {rank} | Word-overlap score: {result['score']}")
        print("Source:", result["passage_id"])
        print("Matched words:", ", ".join(result["matched_words"]))
        print("Passage:", result["text"])

Indexed passages: 9

QUESTION: How many annual leave days do full-time employees receive?

Rank 1 | Word-overlap score: 7
Source: leave_policy:1
Matched words: annual, days, employees, full, leave, receive, time
Passage: Full-time employees receive 20 days of annual leave per calendar year.

Rank 2 | Word-overlap score: 2
Source: leave_policy:2
Matched words: annual, leave
Passage: Annual leave requests must be submitted through the employee portal.

QUESTION: What is the monthly home internet reimbursement amount?

Rank 1 | Word-overlap score: 3
Source: remote_work_policy:3
Matched words: home, internet, reimbursement
Passage: This policy does not specify reimbursement for home internet expenses.

QUESTION: How many annual leave days do contractors receive?

Rank 1 | Word-overlap score: 4
Source: leave_policy:1
Matched words: annual, days, leave, receive
Passage: Full-time employees receive 20 days of annual leave per calendar year.

Rank 2 | Word-overlap score: 2
Source: leave_policy

In [3]:
import json
from datetime import datetime, timezone
from pathlib import Path

# Manual judgments for these three questions.
# These labels are evaluation references, not system predictions.
review_cases = [
    {
        "question": questions[0],
        "expected_behavior": "answer",
        "reference_answer": "20 days of annual leave per calendar year.",
        "review_reason": (
            "leave_policy:1 explicitly states the entitlement "
            "for full-time employees."
        ),
    },
    {
        "question": questions[1],
        "expected_behavior": "abstain",
        "reference_answer": None,
        "review_reason": (
            "remote_work_policy:3 does not provide a reimbursement amount. "
            "This does not imply that reimbursement is zero."
        ),
    },
    {
        "question": questions[2],
        "expected_behavior": "abstain",
        "reference_answer": None,
        "review_reason": (
            "leave_policy:1 covers full-time employees. "
            "It does not establish an entitlement for contractors."
        ),
    },
]

timestamp = datetime.now(timezone.utc)

report = {
    "created_at_utc": timestamp.isoformat(),
    "experiment": "Sentence retrieval using unique word overlap",
    "top_k": 2,
    "indexed_passage_count": len(passages),
    "review_type": "manual reference judgments",
    "limitations": [
        "No LLM or answer-generation component.",
        "No automatic answerability assessment.",
        "Word-overlap scores are not confidence probabilities.",
        "Only three questions against synthetic documents.",
    ],
    "cases": [
        {
            **case,
            "retrieved_passages": retrieve(case["question"], top_k=2),
        }
        for case in review_cases
    ],
}

evidence_dir = Path.cwd() / "evidence"
evidence_dir.mkdir(exist_ok=True)

# A unique filename preserves earlier experiment records.
report_path = evidence_dir / (
    f"retrieval_review_{timestamp.strftime('%Y%m%dT%H%M%S%fZ')}.json"
)

with report_path.open("x", encoding="utf-8") as file:
    json.dump(report, file, indent=2, ensure_ascii=False)

print("Evidence saved:", report_path)
print("Reviewed cases:", len(report["cases"]))
print("Review type:", report["review_type"])

Evidence saved: C:\Users\Khaleel\workplace-ai-operations-lab\evidence\retrieval_review_20261011T113845282789Z.json
Reviewed cases: 3
Review type: manual reference judgments


In [4]:
# Diagnostic probes, separate from our initial three review cases.
retrieval_probes = [
    {
        "question": "What is the yearly PTO allowance for staff?",
        "purpose": (
            "Test a paraphrase of the annual leave question. "
            "Here, PTO means annual leave."
        ),
    },
    {
        "question": "What is the cafeteria breakfast menu?",
        "purpose": "Test a topic absent from all three documents.",
    },
]

for probe in retrieval_probes:
    print("\n" + "=" * 70)
    print("QUESTION:", probe["question"])
    print("PURPOSE:", probe["purpose"])
    print("Query tokens:", sorted(tokenize(probe["question"])))

    results = retrieve(probe["question"], top_k=2)

    if not results:
        print("RESULT: No matching passages.")
    else:
        for result in results:
            print(
                f"\nSource: {result['passage_id']} "
                f"| Score: {result['score']}"
            )
            print("Passage:", result["text"])


QUESTION: What is the yearly PTO allowance for staff?
PURPOSE: Test a paraphrase of the annual leave question. Here, PTO means annual leave.
Query tokens: ['allowance', 'pto', 'staff', 'yearly']
RESULT: No matching passages.

QUESTION: What is the cafeteria breakfast menu?
PURPOSE: Test a topic absent from all three documents.
Query tokens: ['breakfast', 'cafeteria', 'menu']
RESULT: No matching passages.


In [5]:
# Lab-specific vocabulary mappings.
QUERY_EXPANSIONS = {
    "pto": {"annual", "leave"},
    "yearly": {"annual"},
    "staff": {"employees"},
}

def retrieve_expanded(question, top_k=2):
    original_tokens = tokenize(question)
    expanded_tokens = set(original_tokens)

    for token in original_tokens:
        expanded_tokens.update(QUERY_EXPANSIONS.get(token, set()))

    results = []

    for passage in passages:
        matched_words = expanded_tokens & passage["tokens"]

        if matched_words:
            results.append({
                "source": passage["source"],
                "passage_id": passage["passage_id"],
                "text": passage["text"],
                "score": len(matched_words),
                "matched_words": sorted(matched_words),
            })

    results.sort(key=lambda item: (-item["score"], item["passage_id"]))
    return results[:top_k]


# Compare all five questions under both retrieval methods.
comparison_questions = questions + [
    probe["question"] for probe in retrieval_probes
]

for question in comparison_questions:
    baseline = retrieve(question)
    expanded = retrieve_expanded(question)

    print("\n" + "=" * 70)
    print("QUESTION:", question)
    print(
        "Baseline top passage:",
        baseline[0]["passage_id"] if baseline else "None",
    )
    print(
        "Expanded top passage:",
        expanded[0]["passage_id"] if expanded else "None",
    )

    if expanded:
        print("Expanded score:", expanded[0]["score"])
        print("Passage:", expanded[0]["text"])


QUESTION: How many annual leave days do full-time employees receive?
Baseline top passage: leave_policy:1
Expanded top passage: leave_policy:1
Expanded score: 7
Passage: Full-time employees receive 20 days of annual leave per calendar year.

QUESTION: What is the monthly home internet reimbursement amount?
Baseline top passage: remote_work_policy:3
Expanded top passage: remote_work_policy:3
Expanded score: 3
Passage: This policy does not specify reimbursement for home internet expenses.

QUESTION: How many annual leave days do contractors receive?
Baseline top passage: leave_policy:1
Expanded top passage: leave_policy:1
Expanded score: 4
Passage: Full-time employees receive 20 days of annual leave per calendar year.

QUESTION: What is the yearly PTO allowance for staff?
Baseline top passage: None
Expanded top passage: leave_policy:1
Expanded score: 3
Passage: Full-time employees receive 20 days of annual leave per calendar year.

QUESTION: What is the cafeteria breakfast menu?
Baselin

In [6]:
# Additional development checks.
# Expected passage IDs are manually specified retrieval references.
additional_checks = [
    {
        "question": "Where should staff submit annual leave requests?",
        "expected_top_passage": "leave_policy:2",
    },
    {
        "question": "Who approves remote work arrangements?",
        "expected_top_passage": "remote_work_policy:2",
    },
    {
        "question": "What should staff do if a company laptop is lost?",
        "expected_top_passage": "it_support_policy:1",
    },
    {
        "question": "What is the yearly bonus allowance for staff?",
        "expected_top_passage": None,
    },
]

for case in additional_checks:
    print("\n" + "=" * 70)
    print("QUESTION:", case["question"])
    print("Expected top passage:", case["expected_top_passage"])

    for method_name, method in [
        ("Baseline", retrieve),
        ("Expanded", retrieve_expanded),
    ]:
        results = method(case["question"])
        actual_top = results[0]["passage_id"] if results else None

        print(f"\n{method_name} top passage: {actual_top}")
        print("Matches reference:", actual_top == case["expected_top_passage"])

        if results:
            print("Passage:", results[0]["text"])


QUESTION: Where should staff submit annual leave requests?
Expected top passage: leave_policy:2

Baseline top passage: leave_policy:2
Matches reference: True
Passage: Annual leave requests must be submitted through the employee portal.

Expanded top passage: leave_policy:1
Matches reference: False
Passage: Full-time employees receive 20 days of annual leave per calendar year.

QUESTION: Who approves remote work arrangements?
Expected top passage: remote_work_policy:2

Baseline top passage: remote_work_policy:2
Matches reference: True
Passage: The employee's manager must approve the remote work arrangement.

Expanded top passage: remote_work_policy:2
Matches reference: True
Passage: The employee's manager must approve the remote work arrangement.

QUESTION: What should staff do if a company laptop is lost?
Expected top passage: it_support_policy:1

Baseline top passage: it_support_policy:1
Matches reference: True
Passage: Employees must report lost company laptops to the IT service des

In [7]:
import json
from datetime import datetime, timezone
from pathlib import Path

comparison_results = []
summary = {}

for method_name, method in [
    ("baseline", retrieve),
    ("expanded", retrieve_expanded),
]:
    correct = 0

    for case in additional_checks:
        results = method(case["question"], top_k=2)
        actual_top = results[0]["passage_id"] if results else None
        matches = actual_top == case["expected_top_passage"]
        correct += int(matches)

        comparison_results.append({
            "method": method_name,
            "question": case["question"],
            "expected_top_passage": case["expected_top_passage"],
            "actual_top_passage": actual_top,
            "matches_reference": matches,
            "retrieved_passages": results,
        })

    summary[method_name] = {
        "matching_checks": correct,
        "total_checks": len(additional_checks),
    }

timestamp = datetime.now(timezone.utc)

comparison_report = {
    "created_at_utc": timestamp.isoformat(),
    "evaluation_type": "Four manually labeled development checks",
    "top_k": 2,
    "query_expansions": {
        key: sorted(values)
        for key, values in QUERY_EXPANSIONS.items()
    },
    "summary": summary,
    "results": comparison_results,
    "decision": (
        "Retain the original retriever as the baseline. "
        "Keep query expansion experimental because these checks "
        "exposed ranking and irrelevant-match regressions."
    ),
    "limitations": [
        "Small development set; not a held-out evaluation.",
        "Measures top-passage matching, including expected no-result cases.",
        "Does not measure generated-answer correctness.",
    ],
}

evidence_dir = Path.cwd() / "evidence"
evidence_dir.mkdir(exist_ok=True)

comparison_path = evidence_dir / (
    f"query_expansion_comparison_"
    f"{timestamp.strftime('%Y%m%dT%H%M%S%fZ')}.json"
)

with comparison_path.open("x", encoding="utf-8") as file:
    json.dump(comparison_report, file, indent=2, ensure_ascii=False)

for method_name, counts in summary.items():
    print(
        f"{method_name}: {counts['matching_checks']}/"
        f"{counts['total_checks']} checks matched"
    )

print("\nEvidence saved:", comparison_path)

baseline: 4/4 checks matched
expanded: 2/4 checks matched

Evidence saved: C:\Users\Khaleel\workplace-ai-operations-lab\evidence\query_expansion_comparison_20261011T113845354256Z.json


In [8]:
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

# Use the notebook's own Python interpreter to inspect its packages.
result = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"],
    check=True,
    capture_output=True,
    text=True,
)

timestamp = datetime.now(timezone.utc)
evidence_dir = Path.cwd() / "evidence"
evidence_dir.mkdir(exist_ok=True)

snapshot_path = evidence_dir / (
    f"environment_{timestamp.strftime('%Y%m%dT%H%M%S%fZ')}.txt"
)

snapshot = (
    f"# Python: {sys.version.split()[0]}\n"
    "# Platform: Windows\n"
    "# Installed-package snapshot from the project virtual environment\n"
    + result.stdout
)

with snapshot_path.open("x", encoding="utf-8") as file:
    file.write(snapshot)

packages = [
    line for line in result.stdout.splitlines()
    if line and not line.startswith("#")
]

print("Python executable:", sys.executable)
print("Installed packages recorded:", len(packages))
print("Environment snapshot saved:", snapshot_path)

Python executable: C:\Users\Khaleel\workplace-ai-operations-lab\.venv\Scripts\python.exe
Installed packages recorded: 90
Environment snapshot saved: C:\Users\Khaleel\workplace-ai-operations-lab\evidence\environment_20261011T113846868985Z.txt


In [10]:
import difflib

existing_text = readme_path.read_text(encoding="utf-8")

difference = difflib.unified_diff(
    existing_text.splitlines(),
    readme_text.splitlines(),
    fromfile="README.md on disk",
    tofile="Notebook draft",
    lineterm="",
)

print("\n".join(difference))

In [11]:
from pathlib import Path

gitignore_path = Path.cwd() / ".gitignore"

gitignore_text = """\
# Python virtual environments
.venv/
venv/

# Python caches
__pycache__/
*.py[cod]
.pytest_cache/

# Jupyter checkpoints
.ipynb_checkpoints/

# Local environment variables and common credential files
.env
.env.*
!.env.example
*.pem
*.key

# Windows Explorer metadata
Thumbs.db
Desktop.ini
"""

if gitignore_path.exists():
    if gitignore_path.read_text(encoding="utf-8") != gitignore_text:
        raise RuntimeError(
            "An existing .gitignore has different content. Review it first."
        )
    print(".gitignore already contains the expected rules.")
else:
    gitignore_path.write_text(gitignore_text, encoding="utf-8")
    print("Created:", gitignore_path)

print("Synthetic data, notebooks, README, and evidence remain eligible for Git.")

.gitignore already contains the expected rules.
Synthetic data, notebooks, README, and evidence remain eligible for Git.
